In [2]:
# Import all needed libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import sqlite3
import os

In [3]:
# Read 1 of the 9 files to see what it looks like 
# Start from the most important data, which is the GDP Growth
desktop  = os.path.join(os.path.expanduser("~"), "Desktop")
filepath = os.path.join(desktop, "API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_260.csv")

df_gdp = pd.read_csv(filepath, skiprows=4)

# Check how many rows and columns it contains
print("Shape:", df_gdp.shape)

Shape: (266, 71)


In [6]:
# Look at the first 5 rows to understand the structure of the data
df_gdp.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,3.493430,3.212471,1.225112,-23.897990,14.730616,10.636431,7.706798,6.810777,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,0.418937,7.937038,5.623764,4.649241,5.138168,...,2.677524,2.705194,2.030077,-2.817572,4.578772,3.722717,1.931160,2.763839,NaN,NaN
2,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,2.647003,1.189228,3.911603,-2.351101,-20.738839,-6.240172,2.266944,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,1.869593,3.726090,7.038388,5.364089,4.105339,...,2.296349,2.904664,3.281683,-3.730630,2.549691,4.472795,3.662428,4.585674,NaN,NaN
4,Angola,AGO,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.149396,-0.594411,-0.204680,-4.042447,2.102753,4.216003,1.263308,4.423907,NaN,NaN


In [8]:
# See all column names
print(df_gdp.columns.tolist())

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']


In [10]:
# Keep the country columns and year columns only

year_cols = [str(y) for y in range(1960, 2026)]

df_gdp = df_gdp[["Country Name", "Country Code"] + year_cols]

# Rename country name and code
df_gdp = df_gdp.rename(columns={
    "Country Name": "country",
    "Country Code": "country_code"
})

In [12]:
print("Shape:", df_gdp.shape)
print("First 3 rows:")
print(df_gdp.head(5))

Shape: (266, 68)
First 3 rows:
                       country country_code  1960      1961      1962  \
0                        Aruba          ABW   NaN       NaN       NaN   
1  Africa Eastern and Southern          AFE   NaN  0.418937  7.937038   
2                  Afghanistan          AFG   NaN       NaN       NaN   
3   Africa Western and Central          AFW   NaN  1.869593  3.726090   
4                       Angola          AGO   NaN       NaN       NaN   

       1963      1964      1965      1966      1967  ...      2016      2017  \
0       NaN       NaN       NaN       NaN       NaN  ...  1.234335  3.493430   
1  5.623764  4.649241  5.138168  4.827421  5.338839  ...  2.130117  2.677524   
2       NaN       NaN       NaN       NaN       NaN  ...  2.260314  2.647003   
3  7.038388  5.364089  4.105339 -1.513011 -8.966703  ...  0.194316  2.296349   
4       NaN       NaN       NaN       NaN       NaN  ... -1.700530 -0.149396   

       2018      2019       2020       2021      

In [14]:
# Change the format of the data, which is one row per country and per year

df_gdp = df_gdp.melt(
    id_vars    = ["country_code", "country"],  
    var_name   = "year",                        
    value_name = "gdp_growth"                  
)

# Convert year to an integer because this helps us to filter the values more conveniently. This also helps us to do the SQL queries work below.  
df_gdp["year"] = df_gdp["year"].astype(int)

# Convert gdp_growth to number — errors="coerce" turns any text into NaN
df_gdp["gdp_growth"] = pd.to_numeric(df_gdp["gdp_growth"], errors = "coerce")

print("Shape:", df_gdp.shape)
print("First 3 rows:")
print(df_gdp.head(3))

Shape: (17556, 4)
First 3 rows:
  country_code                      country  year  gdp_growth
0          ABW                        Aruba  1960         NaN
1          AFE  Africa Eastern and Southern  1960         NaN
2          AFG                  Afghanistan  1960         NaN


In [16]:
# Check year range
print("Year range:", df_gdp["year"].min(), "to", df_gdp["year"].max())

Year range: 1960 to 2025


In [18]:
# Repeat the same process for all 9 files 

file_map = {
    "API_NY.GDP.PCAP.KD.ZG_DS2_en_csv_v2_376.csv"    : "gdp_growth", # GDP per capita growth (%)
    "API_NE.TRD.GNFS.ZS_DS2_en_csv_v2_101.csv"       : "trade_openness", # Trade as % of GDP
    "API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_287.csv"       : "inflation", # Inflation (%)
    "API_NE.GDI.TOTL.ZS_DS2_en_csv_v2_698.csv"       : "investment", # Investment as % of GDP
    "API_SE.SEC.ENRR_DS2_en_csv_v2_758.csv"           : "school_enrolment", # Secondary school enrolment (%)
    "API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_36.csv"        : "unemployment", # Unemployment rate (%)
    "API_NY.GDP.MKTP.KD.ZG_DS2_en_csv_v2_260.csv"    : "gdp_growth_total", # Total GDP growth (%)
    "API_BX.KLT.DINV.WD.GD.ZS_DS2_en_csv_v2_13.csv"  : "fdi", # Foreign direct investment % GDP
    "API_FM.LBL.BMNY.GD.ZS_DS2_en_csv_v2_5797.csv"   : "broad_money", # Broad money % GDP
}

year_cols = [str(y) for y in range(1960, 2026)]

In [20]:
dfs = []

for filename, varname in file_map.items():
    filepath = os.path.join(desktop, filename)

    # Load file — skip first 4 rows of metadata
    df_temp = pd.read_csv(filepath, skiprows=4)

    # Keep only country columns and year columns
    df_temp = df_temp[["Country Name", "Country Code"] + year_cols]

    # Rename country columns
    df_temp = df_temp.rename(columns={
        "Country Name": "country",
        "Country Code": "country_code"
    })
    
    # Melt from wide to long format
    df_temp = df_temp.melt(
        id_vars    = ["country_code", "country"],
        var_name   = "year",
        value_name = varname
    )

    # Convert year and value columns to numbers
    df_temp["year"]  = df_temp["year"].astype(int)
    df_temp[varname] = pd.to_numeric(df_temp[varname], errors="coerce")

    print(f"  Loaded: {varname:20s} — {len(df_temp)} rows")
    dfs.append(df_temp)

  Loaded: gdp_growth           — 17556 rows
  Loaded: trade_openness       — 17556 rows
  Loaded: inflation            — 17556 rows
  Loaded: investment           — 17556 rows
  Loaded: school_enrolment     — 17556 rows
  Loaded: unemployment         — 17556 rows
  Loaded: gdp_growth_total     — 17556 rows
  Loaded: fdi                  — 17556 rows
  Loaded: broad_money          — 17556 rows


In [22]:
# Start with the first dataframe as the base
df = dfs[0].copy()

# Merge each remaining dataframe one by one
# We match rows on country_code and year
for df_next in dfs[1:]:
    df = pd.merge(
        df,
        df_next.drop(columns=["country"]),  # drop duplicate country name
        on       = ["country_code", "year"],
        how      = "outer",
        validate = "1:1"                     # each country-year should appear once
    )

print("Shape:", df.shape)
print("First 3 rows:")
print(df.head(3))

Shape: (17556, 12)
First 3 rows:
  country_code country  year  gdp_growth  trade_openness  inflation  \
0          ABW   Aruba  1960         NaN             NaN        NaN   
1          ABW   Aruba  1961         NaN             NaN        NaN   
2          ABW   Aruba  1962         NaN             NaN        NaN   

   investment  school_enrolment  unemployment  gdp_growth_total  fdi  \
0         NaN               NaN           NaN               NaN  NaN   
1         NaN               NaN           NaN               NaN  NaN   
2         NaN               NaN           NaN               NaN  NaN   

   broad_money  
0          NaN  
1          NaN  
2          NaN  


In [24]:
# Check the year range for each variable
print("gdp_growth year range:      ", df["year"].min(), "to", df["year"].max())
print("trade_openness year range:  ", df["year"].min(), "to", df["year"].max())
print("inflation year range:       ", df["year"].min(), "to", df["year"].max())
print("investment year range:      ", df["year"].min(), "to", df["year"].max())
print("school_enrolment year range:", df["year"].min(), "to", df["year"].max())
print("unemployment year range:    ", df["year"].min(), "to", df["year"].max())
print("gdp_growth_total year range:", df["year"].min(), "to", df["year"].max())
print("fdi year range:             ", df["year"].min(), "to", df["year"].max())
print("broad_money year range:     ", df["year"].min(), "to", df["year"].max())

gdp_growth year range:       1960 to 2025
trade_openness year range:   1960 to 2025
inflation year range:        1960 to 2025
investment year range:       1960 to 2025
school_enrolment year range: 1960 to 2025
unemployment year range:     1960 to 2025
gdp_growth_total year range: 1960 to 2025
fdi year range:              1960 to 2025
broad_money year range:      1960 to 2025


In [26]:
# Save the merged file to Desktop so we can open it in Excel
out_path = os.path.join(desktop, "pre_cleaned_data.csv")

df.to_csv(out_path, index=False)

In [28]:
# Check the structure of the data after merging all variables into one data frame. 
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17556 entries, 0 to 17555
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   country_code      17556 non-null  object 
 1   country           17556 non-null  object 
 2   year              17556 non-null  int64  
 3   gdp_growth        14133 non-null  float64
 4   trade_openness    10929 non-null  float64
 5   inflation         9066 non-null   float64
 6   investment        10647 non-null  float64
 7   school_enrolment  8362 non-null   float64
 8   unemployment      8211 non-null   float64
 9   gdp_growth_total  14133 non-null  float64
 10  fdi               11800 non-null  float64
 11  broad_money       10881 non-null  float64
dtypes: float64(9), int64(1), object(2)
memory usage: 1.6+ MB
None


In [30]:
# Check the statistical summary of the data
print(df.describe())

               year    gdp_growth  trade_openness     inflation    investment  \
count  17556.000000  14133.000000    10929.000000   9066.000000  10647.000000   
mean    1992.500000      1.938498       72.184411     22.643882     23.646716   
std       19.050914      5.993524       50.721741    317.642980      8.121157   
min     1960.000000    -64.423582        0.020999    -17.640424    -15.678444   
25%     1976.000000     -0.256367       41.507270      2.135129     18.971845   
50%     1992.500000      2.123962       59.479081      4.831985     23.178442   
75%     2009.000000      4.335414       90.171651     10.085932     27.511848   
max     2025.000000    140.490578      863.195099  23773.131774     76.782325   

       school_enrolment  unemployment  gdp_growth_total           fdi  \
count       8362.000000   8211.000000      14133.000000  11800.000000   
mean          63.445131      7.771366          3.672496      4.749404   
std           33.654846      5.540366          6.21

In [32]:
# Check if there are any duplicate data
duplicates = df.duplicated(subset=["country_code", "year"])

print("Duplicate rows:", duplicates.sum())

Duplicate rows: 0


In [34]:
# Drop the rows with missing values
df_clean = df.dropna()

# Check how many rows we have before and after
print("Rows before dropping missing:", df.shape)
print("Rows after dropping missing :", df_clean.shape)

Rows before dropping missing: (17556, 12)
Rows after dropping missing : (2571, 12)


In [36]:
# Keep only rows where the year is between 2000 and 2024 because not all the countries contain data between 1960 and 2024. Therefore, we need to narrow down the time frame.
df_clean = df_clean.query("year >= 2000 and year <= 2024")

# Check the result
print("Year range:", df_clean["year"].min(), "to", df_clean["year"].max())
print("Shape:", df_clean.shape)
print("First 3 rows:")
print(df_clean.head(3))

Year range: 2000 to 2024
Shape: (2184, 12)
First 3 rows:
    country_code                      country  year  gdp_growth  \
106          AFE  Africa Eastern and Southern  2000    0.537870   
107          AFE  Africa Eastern and Southern  2001    0.858579   
108          AFE  Africa Eastern and Southern  2002    1.242191   

     trade_openness  inflation  investment  school_enrolment  unemployment  \
106       46.557169   8.601485   18.983457         27.737129      7.668768   
107       47.955964   5.840354   18.234958         28.789110      7.553885   
108       57.294304   8.763755   17.647979         29.475451      7.511493   

     gdp_growth_total       fdi  broad_money  
106          3.181375  1.534056    35.852469  
107          3.503533  4.774770    39.613793  
108          3.917113  2.453114    41.111370  


In [38]:
# AFE (Africa Eastern and Southern) is a regional code, not a real country. Therefore, we need to remove it from the data.
df_clean = df_clean.query("country_code != 'AFE'")

# Check the result
print("Countries remaining:", df_clean["country_code"].nunique())
print("Shape:", df_clean.shape)

Countries remaining: 139
Shape: (2159, 12)


In [40]:
# Save the final cleaned file to Desktop
out_path = os.path.join(desktop, "cleaned_data.csv")

df_clean.to_csv(out_path, index=False)

In [42]:
# Read the cleaned data file in the desktop
desktop  = os.path.join(os.path.expanduser("~"), "Desktop")
filepath = os.path.join(desktop, "cleaned_data.csv")

df = pd.read_csv(filepath)

In [44]:
conn = sqlite3.connect(":memory:")

In [48]:
df.to_sql("world", conn, index=False, if_exists = "replace")

2159

In [50]:
output1 = pd.read_sql("""
    SELECT
        -- Full sample (2000-2024)
        ROUND(AVG(gdp_growth),        2) AS avg_gdp_growth,
        ROUND(MIN(gdp_growth),        2) AS min_gdp_growth,
        ROUND(MAX(gdp_growth),        2) AS max_gdp_growth,

        ROUND(AVG(trade_openness),    2) AS avg_trade_openness,
        ROUND(MIN(trade_openness),    2) AS min_trade_openness,
        ROUND(MAX(trade_openness),    2) AS max_trade_openness,

        ROUND(AVG(inflation),         2) AS avg_inflation,
        ROUND(MIN(inflation),         2) AS min_inflation,
        ROUND(MAX(inflation),         2) AS max_inflation,

        ROUND(AVG(investment),        2) AS avg_investment,
        ROUND(MIN(investment),        2) AS min_investment,
        ROUND(MAX(investment),        2) AS max_investment,

        ROUND(AVG(school_enrolment),  2) AS avg_school_enrolment,
        ROUND(MIN(school_enrolment),  2) AS min_school_enrolment,
        ROUND(MAX(school_enrolment),  2) AS max_school_enrolment,

        ROUND(AVG(unemployment),      2) AS avg_unemployment,
        ROUND(MIN(unemployment),      2) AS min_unemployment,
        ROUND(MAX(unemployment),      2) AS max_unemployment,

        ROUND(AVG(fdi),               2) AS avg_fdi,
        ROUND(MIN(fdi),               2) AS min_fdi,
        ROUND(MAX(fdi),               2) AS max_fdi,

        ROUND(AVG(broad_money),       2) AS avg_broad_money,
        ROUND(MIN(broad_money),       2) AS min_broad_money,
        ROUND(MAX(broad_money),       2) AS max_broad_money

    FROM world
    WHERE year BETWEEN 2000 AND 2024
""", conn)

In [52]:
summary = pd.DataFrame({
    "Indicator"  : [
        "GDP Growth (%)", "Trade Openness (% GDP)", "Inflation (%)",
        "Investment (% GDP)", "School Enrolment (%)",
        "Unemployment (%)", "FDI (% GDP)", "Broad Money (% GDP)"
    ],
    "Mean" : [
        output1["avg_gdp_growth"][0],    output1["avg_trade_openness"][0],
        output1["avg_inflation"][0],     output1["avg_investment"][0],
        output1["avg_school_enrolment"][0], output1["avg_unemployment"][0],
        output1["avg_fdi"][0],           output1["avg_broad_money"][0]
    ],
    "Min"  : [
        output1["min_gdp_growth"][0],    output1["min_trade_openness"][0],
        output1["min_inflation"][0],     output1["min_investment"][0],
        output1["min_school_enrolment"][0], output1["min_unemployment"][0],
        output1["min_fdi"][0],           output1["min_broad_money"][0]
    ],
    "Max"  : [
        output1["max_gdp_growth"][0],    output1["max_trade_openness"][0],
        output1["max_inflation"][0],     output1["max_investment"][0],
        output1["max_school_enrolment"][0], output1["max_unemployment"][0],
        output1["max_fdi"][0],           output1["max_broad_money"][0]
    ]
})

In [66]:
print("Output 1: Summary Statistics — Key Macroeconomic Indicators (2000–2024)")
(summary.style.background_gradient(cmap = "Blues", subset = ["Mean"]).format({"Mean": "{:.2f}", "Min": "{:.2f}", "Max": "{:.2f}"})
.set_caption("Table 1: Summary Statistics — Key Macroeconomic Indicators (2000–2024)"))

Output 1: Summary Statistics — Key Macroeconomic Indicators (2000–2024)


,Indicator,Mean,Min,Max
0,GDP Growth (%),2.43,-55.29,74.92
1,Trade Openness (% GDP),81.60,15.28,442.62
2,Inflation (%),5.70,-16.86,133.49
3,Investment (% GDP),24.79,5.50,70.33
4,School Enrolment (%),79.08,6.23,159.11
5,Unemployment (%),7.39,0.10,37.32
6,FDI (% GDP),3.96,-40.11,105.64
7,Broad Money (% GDP),79.73,5.54,13083.81
